### RAG application by using typesense


In [1]:
import typesense

In [ ]:
client=typesense.Client({
    'nodes':[{
        'host': '0w853kv7nfiqbz2lp-1.a1.typesense.net',
        'port':'443',
        'protocol':'https'

    }],
    'api_key':'YOUR_TYPESENSE_API_KEY',
    'connection_timeout_seconds':2
    

})

books_schema={
    "name": "books",
    "fields": [
        {"name": "title", "type": "string"},
        {"name": "authors", "type": "string[]",'facet':True},
        {"name": "publication_year", "type": "int32",'facet':True},
        {"name": "id", "type": "string"},
        {"name": "average_rating", "type": "float"},
        {"name": "image_url", "type": "string"},
        {"name": "ratings_count", "type": "int32"}
    ],
    'default_sorting_field':"ratings_count"
}
print(client.collections.create(books_schema))

{'created_at': 1774795154, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'average_rating', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'float'}, {'facet': False, 'index': 

In [6]:
client

In [14]:
with open("books.jsonl",'r',encoding='utf-8') as json_file:
    data=json_file.read()
    client.collections['books'].documents.import_(data)


In [18]:
search_parameters={
    'q':' The Man in the High Castle',
    'query_by':'title,authors',
    'sort_by':'ratings_count:desc',
    'filter_by':'publication_year:>1950 && publication_year:<2000'

}
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['Philip K. Dick'],
    'average_rating': 3.66,
    'id': '974',
    'image_url': 'https://images.gr-assets.com/books/1448756803m/216363.jpg',
    'publication_year': 1962,
    'ratings_count': 84180,
    'title': 'The Man in the High Castle'},
   'highlight': {'title': {'matched_tokens': ['The',
      'Man',
      'in',
      'the',
      'High',
      'Castle'],
     'snippet': '<mark>The</mark> <mark>Man</mark> <mark>in</mark> <mark>the</mark> <mark>High</mark> <mark>Castle</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['The', 'Man', 'in', 'the', 'High', 'Castle'],
     'snippet': '<mark>The</mark> <mark>Man</mark> <mark>in</mark> <mark>the</mark> <mark>High</mark> <mark>Castle</mark>'}],
   'text_match': 3472336863744755833,
   'text_match_info': {'best_field_score': '6627123986433',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '3472336863

### langchain + typesense + GroqLLM + RAG Application

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

e:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os 
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY', 'YOUR_GROQ_API_KEY')

In [6]:
loader=TextLoader('test.txt')
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
docs=text_splitter.split_documents(documents)

embeddings=HuggingFaceEmbeddings()


C:\Users\jay\AppData\Local\Temp\ipykernel_15536\1956374023.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings()
C:\Users\jay\AppData\Local\Temp\ipykernel_15536\1956374023.py:6: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings=HuggingFaceEmbeddings()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9440.75it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+-----

In [ ]:
doc_search=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host":"0w853kv7nfiqbz2lp-1.a1.typesense.net",
        "port":"443",
        "protocol":"https",
        "typesense_api_key":"YOUR_TYPESENSE_API_KEY",
        "typesense_collection_name":"lang-chain"
    },
)

In [8]:
query='what is ai '
docs=doc_search.similarity_search(query)
print(docs[0].page_content)

ARTIFICIAL INTELLIGENCE (AI) - DETAILED NOTES

1. Introduction to AI
Artificial Intelligence (AI) is a branch of computer science that aims to create machines capable of performing tasks that typically require human intelligence. These tasks include learning, reasoning, problem-solving, perception, and language understanding.

AI systems can analyze data, recognize patterns, and make decisions with minimal human intervention.

2. Types of AI
AI can be classified into three main types:

a) Narrow AI (Weak AI)
- Designed for a specific task
- Examples: Voice assistants, recommendation systems

b) General AI (Strong AI)
- Can perform any intellectual task like humans
- Still under research

c) Super AI
- Surpasses human intelligence
- Theoretical concept

3. Applications of AI
AI is widely used in various fields:


In [10]:
retriever=doc_search.as_retriever()
retriever


VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000001EDB6B49550>, search_kwargs={})